In [40]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

# --- Step 1: Load filtered test and training data ---
train_df = pd.read_csv('filtered_train_data.csv')
test_df = pd.read_csv('filtered_test_data.csv')

# Save important meta columns before processing
test_meta = test_df[['race_id', 'won']]

# --- Step 2: One-hot encode 'race_class' using the same categories ---
race_class_categories = sorted(train_df['race_class'].unique())
train_df = pd.get_dummies(train_df, columns=['race_class'], drop_first=False)
test_df = pd.get_dummies(test_df, columns=['race_class'], drop_first=False)

# --- Step 3: Manually define feature columns ---
feature_cols = ['declared_weight', 'actual_weight', 'win_odds', 'distance'] + \
               [f'race_class_{c}' for c in race_class_categories]

# --- Step 4: Prepare feature matrices ---
X_train = train_df[feature_cols]
y_train = train_df['finish_time']

X_test = test_df.reindex(columns=feature_cols, fill_value=0)

# --- Step 5: Standardize features ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Step 6: Train model ---
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# --- Step 7: Predict on test set ---
test_df['predicted_finish_time'] = model.predict(X_test_scaled)

# Recover race_id and won
test_df['race_id'] = test_meta['race_id'].values
test_df['won'] = test_meta['won'].values

# --- Step 8: Group by race_id and evaluate ---
def predict_winner(group):
    predicted_winner_index = group['predicted_finish_time'].idxmin()
    return group.loc[predicted_winner_index, 'won'] == 1

results = test_df.groupby('race_id').apply(predict_winner)

# --- Step 9: Calculate accuracy ---
accuracy = results.mean()
print(f"Prediction Accuracy: {accuracy:.4f} ({results.sum()} correct out of {len(results)} races)")


Prediction Accuracy: 0.3053 (290 correct out of 950 races)


C:\Users\siuts\AppData\Local\Temp\ipykernel_48444\2394679268.py:48: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  results = test_df.groupby('race_id').apply(predict_winner)
